# Notebook 01 -- EDA, Dimensionality Reduction and Clustering

This notebook answers three questions about the cleaned UNSW-NB15 training partition, before any
supervised model is fitted:

1. **What does the data look like?** Distributions, skew, categorical balance, class imbalance.
2. **How much of the feature space is redundant?** Correlation, multicollinearity (VIF) and PCA
   quantify how many real degrees of freedom the feature set actually has.
3. **Does unsupervised structure exist at all?** K-Means and DBSCAN on the PCA-reduced space are
   compared against the true attack categories, strictly post hoc.

Every result that depends on the `sttl` / `ct_state_ttl` testbed shortcut pair is reported twice --
with and without that pair -- per the project's data rules.

**How to re-run this notebook:**

```
uv run jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 notebooks/01_eda_reduction_clustering.ipynb
```

All exported tables, figures, metrics and notes are written to `results/eda_reduction_clustering/`.

In [1]:
import json
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression
from sklearn.metrics import silhouette_score, calinski_harabasz_score

from nids.columns import (
    feature_columns,
    TTL_SHORTCUT_COLUMNS,
    CATEGORICAL_COLUMNS,
    SKEWED_COLUMNS,
    numeric_feature_columns,
    skewed_feature_columns,
)
from nids.data import load_clean_partitions
from nids.preprocessing import build_preprocessor, build_preprocessor_pair
from nids.sampling import stratified_subsample
from nids.results import ResultsWriter
from nids.validation import validate_raw_data

In [2]:
%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 100,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.facecolor": "white",
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.frameon": True,
})
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)

In [3]:
NOTEBOOK_ID = "eda_reduction_clustering"
SEED = 42
VARIANTS = {"with_ttl": True, "without_ttl": False}
K_RANGE = range(2, 13)
VARIANCE_THRESHOLDS = (0.90, 0.95)
KEY_FEATURES = ("dur", "sbytes", "dbytes", "rate", "sload", "dload")
MAX_ROWS = 10_000
FLOOR = 50
VARIANT_COLORS = {"with_ttl": "#1f77b4", "without_ttl": "#d62728"}
VARIANT_LINESTYLES = {"with_ttl": "-", "without_ttl": "--"}
VARIANT_LABELS = {"with_ttl": "With TTL shortcut", "without_ttl": "Without TTL shortcut"}

## Why raw validation runs first

The raw UNSW-NB15 files are validated before anything else happens. If either file has been
truncated, replaced, or has drifted in shape or schema, this notebook must fail at this cell -- not
silently produce wrong numbers halfway through section 3.

In [4]:
report = validate_raw_data()
print(report.render())
assert report.ok

[OK] UNSW_NB15_training-set.csv
    rows=175341 (expected 175341), columns=45 (expected 45)
    columns_match=True, numeric_dtypes_ok=True, null_cells=0
[OK] UNSW_NB15_testing-set.csv
    rows=82332 (expected 82332), columns=45 (expected 45)
    columns_match=True, numeric_dtypes_ok=True, null_cells=0

OK


In [5]:
cleaned = load_clean_partitions()
train = cleaned.train
del cleaned
assert "cleaned" not in globals()

CLASS_ORDER = sorted(train["attack_cat"].unique().tolist())
if len(CLASS_ORDER) <= 10:
    CLASS_COLORS = {c: plt.get_cmap("tab10")(i) for i, c in enumerate(CLASS_ORDER)}
else:
    CLASS_COLORS = {c: plt.get_cmap("tab20")(i) for i, c in enumerate(CLASS_ORDER)}
assert len(CLASS_ORDER) <= 20

print(f"train shape: {train.shape}")
print(f"classes ({len(CLASS_ORDER)}): {CLASS_ORDER}")

train shape: (107740, 41)
classes (10): ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']


## The `attack_cat` boundary

`attack_cat` legitimately appears four times in this notebook:

1. Descriptive statistics -- the class-distribution table, the sorted bar chart, and boxplot
   grouping (section 1).
2. As the stratification key for the subsampler, so silhouette and DBSCAN evaluate a
   class-representative sample.
3. As the post-hoc colouring for the PC1/PC2 scatter -- applied only after the projection is
   already computed.
4. As the post-hoc cross-tabulation against cluster labels in cluster profiling (section 5).

Three structural guarantees keep it out of every fit:

- **Allow-list.** The feature columns come from an explicit allow-list of 39 names, never from
  "all columns minus the targets" -- a skipped subtraction step cannot reintroduce a target.
- **Remainder dropped.** The fitted preprocessing pipeline discards every column outside that
  allow-list, even if a wider frame were passed to it by mistake.
- **No fit against a target anywhere.** PCA, K-Means and DBSCAN in this notebook are all fitted on
  the feature matrix alone; none of them ever receives a target argument.

Two runtime assertion cells below prove this boundary inside the executed output, rather than only
claiming it in prose.

In [6]:
for include_ttl in (True, False):
    selected = feature_columns(include_ttl)
    assert "attack_cat" not in selected and "label" not in selected
    assert "row_position" not in selected
assert len(feature_columns(True)) == 39 and len(feature_columns(False)) == 37
assert "cleaned" not in globals()
print("Label boundary and test-partition guards passed.")

Label boundary and test-partition guards passed.


In [7]:
METRICS: dict[str, tuple[int | float | str, str]] = {}
NOTES: dict[str, str] = {}


@dataclass(frozen=True, slots=True)
class VariantSpace:
    key: str
    include_ttl: bool
    columns: list[str]
    pipeline: object
    names: list[str]
    design: np.ndarray
    modelled: pd.DataFrame
    blocks: dict[str, list[str]]


def build_variant_space(key: str, include_ttl: bool, pipe) -> VariantSpace:
    """Assemble one TTL variant's fitted design matrix and derived views."""
    columns = feature_columns(include_ttl)
    names = pipe.get_feature_names_out().tolist()
    design = pipe.transform(train[columns])
    numeric_names = numeric_feature_columns(include_ttl)
    skewed_names = skewed_feature_columns(include_ttl)
    modelled_set = set(numeric_names) | set(skewed_names)
    onehot_names = [n for n in names if n not in modelled_set]
    blocks = {"numeric": numeric_names, "skewed": skewed_names, "onehot": onehot_names}
    modelled_names = [n for n in names if n in modelled_set]
    design_frame = pd.DataFrame(design, columns=names)
    modelled = design_frame[modelled_names]
    return VariantSpace(
        key=key,
        include_ttl=include_ttl,
        columns=columns,
        pipeline=pipe,
        names=names,
        design=design,
        modelled=modelled,
        blocks=blocks,
    )


def variance_inflation_factors(frame: pd.DataFrame) -> pd.DataFrame:
    """VIF = 1 / (1 - R2) per column, regressed on every other column."""
    spread = frame.std(ddof=0)
    constant = [c for c in frame.columns if not np.isfinite(spread[c]) or spread[c] <= 0.0]
    usable = [c for c in frame.columns if c not in constant]

    rows = [
        {"feature": c, "r_squared": float("nan"), "vif": float("nan"), "status": "zero_variance"}
        for c in constant
    ]
    design = frame[usable]
    for column in usable:
        others = design[[c for c in usable if c != column]]
        target = design[column]
        r2 = float(LinearRegression(n_jobs=-1).fit(others, target).score(others, target))
        infinite = (not np.isfinite(r2)) or r2 >= 1.0
        rows.append({
            "feature": column,
            "r_squared": r2,
            "vif": float("inf") if infinite else 1.0 / (1.0 - r2),
            "status": "perfect_collinearity" if infinite else "ok",
        })
    return pd.DataFrame(rows)


def fix_component_signs(components: np.ndarray, scores: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Force each component's largest-magnitude loading to be positive.

    Ties on magnitude are broken by the smallest feature index, because
    np.argmax returns the first maximum.
    """
    components = components.copy()
    scores = scores.copy()
    pivots = np.argmax(np.abs(components), axis=1)
    signs = np.sign(components[np.arange(components.shape[0]), pivots])
    signs[signs == 0.0] = 1.0
    components *= signs[:, None]
    scores *= signs[None, :]
    return components, scores


def select_eps(sample: np.ndarray, min_samples: int) -> tuple[float, np.ndarray, str]:
    """Normalised-chord eps rule with three degenerate-case guards."""
    neighbours = NearestNeighbors(n_neighbors=min_samples, n_jobs=-1).fit(sample)
    distances, _ = neighbours.kneighbors(sample)
    kdist = np.sort(distances[:, min_samples - 1])

    span = kdist[-1] - kdist[0]
    if span == 0:
        return float(np.median(kdist)), kdist, "flat_curve_median"

    m = kdist.size
    x = np.arange(m, dtype=np.float64) / (m - 1)
    y = (kdist - kdist[0]) / span
    dx, dy = x[-1] - x[0], y[-1] - y[0]
    perpendicular = np.abs(dy * (x - x[0]) - dx * (y - y[0])) / np.hypot(dx, dy)
    eps = float(kdist[int(np.argmax(perpendicular))])

    if eps == 0:
        positive = kdist[kdist > 0]
        if positive.size == 0:
            return float("nan"), kdist, "smallest_positive"
        return float(positive.min()), kdist, "smallest_positive"

    return eps, kdist, "chord"


def record_metric(name: str, value: int | float | str, description: str) -> None:
    if name in METRICS:
        raise ValueError(f"Metric {name!r} already registered.")
    METRICS[name] = (value, description)
    writer.add_metric(name, value, description)


def record_note(note_id: str, text: str) -> None:
    if note_id in NOTES:
        raise ValueError(f"Note {note_id!r} already registered.")
    NOTES[note_id] = text
    writer.add_note(note_id, text)


def new_figure(nrows: int, ncols: int, *, figsize: tuple[float, float]):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    return fig, axes

In [8]:
pipelines = build_preprocessor_pair()
spaces: dict[str, VariantSpace] = {}
for key, include_ttl in VARIANTS.items():
    pipe = pipelines[key].fit(train[feature_columns(include_ttl)])
    spaces[key] = build_variant_space(key, include_ttl, pipe)

for key, space in spaces.items():
    print(key, "design:", space.design.shape, "modelled:", space.modelled.shape)

with_ttl design: (107740, 53) modelled: (107740, 36)
without_ttl design: (107740, 51) modelled: (107740, 34)


In [9]:
for key, space in spaces.items():
    assert list(space.pipeline.feature_names_in_) == feature_columns(space.include_ttl)
    assert not ({"attack_cat", "label"} & set(space.names))
    assert space.design.shape[0] == len(train)
print("Fitted design matrices carry no target column, in either TTL variant.")

Fitted design matrices carry no target column, in either TTL variant.


In [10]:
index_frame = pd.DataFrame({
    "attack_cat": train["attack_cat"].to_numpy(),
    "row_position": np.arange(len(train), dtype=np.int64),
})
sub_default = stratified_subsample(index_frame, max_rows=MAX_ROWS, floor=FLOOR, seed=SEED)
sub_proportional = stratified_subsample(index_frame, max_rows=MAX_ROWS, floor=0, seed=SEED)
positions = sub_default.data["row_position"].to_numpy()
proportional_positions = sub_proportional.data["row_position"].to_numpy()

assert "row_position" not in feature_columns(True)
print(f"positions: {positions.shape[0]} rows (dtype={positions.dtype})")
print(f"proportional_positions: {proportional_positions.shape[0]} rows")

positions: 10000 rows (dtype=int64)
proportional_positions: 10000 rows


In [11]:
writer = ResultsWriter(NOTEBOOK_ID)
print(f"writer.directory: {writer.directory}")

record_metric(
    "train_rows",
    len(train),
    "Rows in the cleaned training partition analysed by this notebook.",
)
record_metric(
    "attack_cat_classes",
    int(train["attack_cat"].nunique()),
    "Distinct attack categories present in the cleaned training partition.",
)
record_metric(
    "feature_columns_with_ttl",
    len(feature_columns(True)),
    "Size of the feature allow-list including the TTL shortcut pair.",
)
record_metric(
    "feature_columns_without_ttl",
    len(feature_columns(False)),
    "Size of the feature allow-list excluding sttl and ct_state_ttl.",
)
record_metric(
    "silhouette_sample_rows",
    len(sub_default.data),
    "Rows in the floor=50 stratified subsample used for silhouette and DBSCAN.",
)
record_metric(
    "silhouette_sensitivity_sample_rows",
    len(sub_proportional.data),
    "Rows in the floor=0 proportional subsample used for the sensitivity silhouette.",
)

writer.directory: /home/pato/Desktop/machine_learning_project/results/eda_reduction_clustering


## Section 1 -- Descriptive statistics

This section describes the cleaned training partition on its own terms. It deliberately overlaps
some of what `results/data_cleaning/` already shows, because the output contract requires every
producer folder to be interpretable in isolation: the cleaning report shows before-versus-after
cleaning, while this notebook describes the cleaned training partition only.

In [12]:
roles = []
for position, column in enumerate(train.columns):
    if column in ("attack_cat", "label"):
        role = "target"
    elif column in CATEGORICAL_COLUMNS:
        role = "feature_categorical"
    elif column in SKEWED_COLUMNS:
        role = "feature_skewed"
    else:
        role = "feature_numeric"
    roles.append({
        "position": position,
        "column": column,
        "dtype": str(train[column].dtype),
        "non_null": int(train[column].notna().sum()),
        "role": role,
    })
partition_shape_and_dtypes = pd.DataFrame(roles)
writer.add_table(
    partition_shape_and_dtypes,
    name="partition_shape_and_dtypes",
    title="Column inventory of the cleaned training partition",
    description=(
        "Every kept column in canonical order with its dtype, non-null count and role "
        "(feature_numeric, feature_skewed, feature_categorical, target)."
    ),
    sort_by=["position"],
)
partition_shape_and_dtypes.head(10)

,position,column,dtype,non_null,role
0,0,dur,float64,107740,feature_skewed
1,1,proto,str,107740,feature_categorical
2,2,service,str,107740,feature_categorical
3,3,state,str,107740,feature_categorical
4,4,spkts,int64,107740,feature_skewed
5,5,dpkts,int64,107740,feature_skewed
6,6,sbytes,int64,107740,feature_skewed
7,7,dbytes,int64,107740,feature_skewed
8,8,rate,float64,107740,feature_skewed
9,9,sttl,int64,107740,feature_numeric


In [13]:
numeric_all = [c for c in train.columns if c not in CATEGORICAL_COLUMNS and c not in ("attack_cat", "label")]
rows = []
for feature in numeric_all:
    series = train[feature]
    mean = float(series.mean())
    median = float(series.median())
    rows.append({
        "feature": feature,
        "count": int(series.count()),
        "mean": mean,
        "median": median,
        "std": float(series.std()),
        "min": float(series.min()),
        "q25": float(series.quantile(0.25)),
        "q75": float(series.quantile(0.75)),
        "max": float(series.max()),
        "skew": float(series.skew()),
        "mean_median_ratio": (mean / median) if median != 0 else float("nan"),
    })
numeric_summary_mean_vs_median = pd.DataFrame(rows)
writer.add_table(
    numeric_summary_mean_vs_median,
    name="numeric_summary_mean_vs_median",
    title="Central tendency and spread of the numeric features",
    description=(
        "Raw-unit summary contrasting mean against median to expose right skew. "
        "mean_median_ratio is null where the median is zero."
    ),
    sort_by=["feature"],
)
numeric_summary_mean_vs_median.head(10)

,feature,count,mean,median,std,min,q25,q75,max,skew,mean_median_ratio
0,dur,107740,1.355982e+00,0.305082,5.521568e+00,0.0,0.001701,9.711053e-01,5.999999e+01,8.354809,4.444656
1,spkts,107740,3.019046e+01,10.000000,1.734179e+02,1.0,4.000000,2.200000e+01,9.616000e+03,32.012743,3.019046
2,dpkts,107740,3.005909e+01,8.000000,1.365459e+02,0.0,2.000000,1.800000e+01,1.097400e+04,30.006802,3.757386
3,sbytes,107740,1.382541e+04,952.000000,2.226347e+05,28.0,501.500000,2.542000e+03,1.296523e+07,35.593752,14.522489
4,dbytes,107740,2.373248e+04,354.000000,1.786208e+05,0.0,178.000000,3.080000e+03,1.465555e+07,32.131877,67.040896
5,rate,107740,3.869093e+04,87.279626,1.166318e+05,0.0,26.233821,4.020464e+03,1.000000e+06,5.355056,443.298584
6,sttl,107740,1.430522e+02,62.000000,1.077762e+02,0.0,31.000000,2.540000e+02,2.550000e+02,0.039823,2.307294
7,dttl,107740,1.216000e+02,29.000000,1.160158e+02,0.0,29.000000,2.520000e+02,2.540000e+02,0.217041,4.193105
8,sload,107740,4.968117e+07,87628.792970,2.114064e+08,0.0,9317.050537,1.115275e+06,5.988000e+09,9.403655,566.950342
9,dload,107740,1.089236e+06,7912.984863,3.013781e+06,0.0,2113.082825,5.462244e+05,2.242273e+07,3.546844,137.651674


In [14]:
rows = []
for column in CATEGORICAL_COLUMNS:
    counts = train[column].value_counts(dropna=False)
    shares = counts / len(train)
    for category, rows_count in counts.items():
        rows.append({
            "column": column,
            "category": str(category),
            "rows": int(rows_count),
            "share": float(shares[category]),
        })
categorical_value_counts = pd.DataFrame(rows)
writer.add_table(
    categorical_value_counts,
    name="categorical_value_counts",
    title="Categorical feature frequencies",
    description="proto / service / state frequencies on the cleaned training partition.",
    sort_by=["column", "category"],
)
categorical_value_counts.head(10)

,column,category,rows,share
0,proto,tcp,76293,0.708121
1,proto,udp,23027,0.213727
2,proto,unas,2582,0.023965
3,proto,ospf,775,0.007193
4,proto,arp,633,0.005875
5,proto,sctp,316,0.002933
6,proto,any,93,0.000863
7,proto,gre,63,0.000585
8,proto,rsvp,62,0.000575
9,proto,ipv6,59,0.000548


In [15]:
counts = train["attack_cat"].value_counts()
shares = counts / len(train)
attack_cat_distribution = pd.DataFrame({
    "attack_cat": counts.index,
    "rows": counts.to_numpy(dtype=np.int64),
    "share": shares.to_numpy(),
})
writer.add_table(
    attack_cat_distribution,
    name="attack_cat_distribution",
    title="Attack category balance of the cleaned training partition",
    description="Row count and share per class, describing the label without using it as an input.",
    sort_by=["attack_cat"],
)
attack_cat_distribution.head(10)

,attack_cat,rows,share
0,Normal,51890,0.481622
1,Exploits,19844,0.184184
2,Fuzzers,16150,0.149898
3,Reconnaissance,7522,0.069816
4,Generic,4181,0.038806
5,DoS,3806,0.035326
6,Analysis,1594,0.014795
7,Backdoor,1535,0.014247
8,Shellcode,1091,0.010126
9,Worms,127,0.001179


## Reading the class imbalance

The bar chart below uses a logarithmic row axis because the largest attack category outnumbers the
smallest by roughly two orders of magnitude. A linear axis would make every minority class
invisible.

In [16]:
counts = train["attack_cat"].value_counts().sort_values(ascending=True)
fig, ax = new_figure(1, 1, figsize=(10, 6))
colors = [CLASS_COLORS[c] for c in counts.index]
ax.barh(counts.index, counts.to_numpy(), color=colors)
ax.set_xscale("log")
ax.set_xlabel("Rows (log scale)")
ax.set_ylabel("Attack category")
ax.set_title("Attack category distribution in the cleaned training partition")
fig.tight_layout()
writer.add_figure(
    fig,
    name="attack_cat_distribution_bars",
    title="Attack category distribution in the cleaned training partition",
    description=(
        "Sorted horizontal bars on a logarithmic row axis, because the largest class outnumbers "
        "the smallest by roughly two orders of magnitude."
    ),
)
plt.close(fig)

## Right skew and the log1p rationale

Several key volumetric and rate features are heavily right-skewed: a small number of very large
values stretch the distribution far past its bulk. The preprocessing pipeline applies a log1p
transform to these columns before scaling, which is why this notebook shows both the raw and the
transformed histograms below.

The ECDF panel that follows shows the *same six curves* under two different axis
parameterisations -- not two different distributions. An empirical cumulative distribution is
invariant under any strictly increasing transform, and log1p is strictly increasing, so re-spacing
the x-axis cannot change the curve's shape, only how it is read. That is a stated finding here, not
a bug to fix.

In [17]:
fig, axes = new_figure(2, 3, figsize=(12, 7))
for ax, feature in zip(axes.flat, KEY_FEATURES):
    ax.hist(train[feature].to_numpy(), bins=50, color="steelblue")
    ax.set_yscale("log")
    ax.set_title(feature)
    ax.set_xlabel("Value")
    ax.set_ylabel("Rows (log scale)")
fig.suptitle("Key feature distributions in raw units")
fig.tight_layout()
writer.add_figure(
    fig,
    name="key_feature_histograms_raw",
    title="Key feature distributions in raw units",
    description=(
        "Histogram grid for six volumetric and rate features in raw units, with a logarithmic "
        "count axis so the right tail is visible."
    ),
)
plt.close(fig)

In [18]:
fig, axes = new_figure(2, 3, figsize=(12, 7))
for ax, feature in zip(axes.flat, KEY_FEATURES):
    values = np.log1p(train[feature].to_numpy())
    ax.hist(values, bins=50, color="steelblue")
    ax.set_title(feature)
    ax.set_xlabel("log1p(value)")
    ax.set_ylabel("Rows")
fig.suptitle("Key feature distributions after log1p")
fig.tight_layout()
writer.add_figure(
    fig,
    name="key_feature_histograms_log1p",
    title="Key feature distributions after log1p",
    description=(
        "The same six features after the log1p transform the preprocessing pipeline applies to "
        "skewed columns."
    ),
)
plt.close(fig)

In [19]:
fig, axes = new_figure(1, 2, figsize=(13, 6))
palette = plt.get_cmap("tab10")
for i, feature in enumerate(KEY_FEATURES):
    raw_values = np.sort(train[feature].to_numpy())
    proportion = np.arange(1, raw_values.size + 1) / raw_values.size
    axes[0].step(raw_values, proportion, where="post", color=palette(i), label=feature)
    axes[1].step(np.log1p(raw_values), proportion, where="post", color=palette(i), label=feature)

axes[0].set_xscale("symlog", linthresh=1)
axes[0].set_xlabel("Value (symlog scale, linear below 1)")
axes[0].set_ylabel("Cumulative proportion of rows")
axes[0].set_title("Raw value axis")
axes[1].set_xlabel("log1p(value)")
axes[1].set_ylabel("Cumulative proportion of rows")
axes[1].set_title("log1p(value) axis")
axes[1].legend(loc="lower right", fontsize=8)
fig.suptitle("Empirical cumulative distributions: raw units versus log1p")
fig.tight_layout()
writer.add_figure(
    fig,
    name="key_feature_ecdf_raw_vs_log1p",
    title="Empirical cumulative distributions: raw units versus log1p",
    description=(
        "Two panels showing the same six empirical cumulative distributions under a raw symlog "
        "axis and a log1p linear axis. The curves are identical by construction: log1p is "
        "strictly increasing, so it re-spaces the axis without changing the distribution."
    ),
)
plt.close(fig)

In [20]:
fig, axes = new_figure(3, 2, figsize=(12, 14))
for ax, feature in zip(axes.flat, KEY_FEATURES):
    data_by_class = [train.loc[train["attack_cat"] == c, feature].to_numpy() for c in CLASS_ORDER]
    box = ax.boxplot(
        data_by_class,
        vert=False,
        tick_labels=CLASS_ORDER,
        showfliers=False,
        whis=(1, 99),
        patch_artist=True,
    )
    for patch, cls in zip(box["boxes"], CLASS_ORDER):
        patch.set_facecolor(CLASS_COLORS[cls])
    ax.set_xscale("symlog", linthresh=1)
    ax.set_xlabel("Value (symlog scale, linear below 1)")
    ax.set_title(feature)
fig.suptitle("Key feature spread by attack category")
fig.tight_layout()
writer.add_figure(
    fig,
    name="key_feature_boxplots_by_attack_cat",
    title="Key feature spread by attack category",
    description=(
        "Horizontal boxplots per attack category on a symlog value axis. Whiskers are the 1st "
        "and 99th percentiles; individual outliers are omitted."
    ),
)
plt.close(fig)

/tmp/ipykernel_61445/2848139186.py:4: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  box = ax.boxplot(
/tmp/ipykernel_61445/2848139186.py:4: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  box = ax.boxplot(
/tmp/ipykernel_61445/2848139186.py:4: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  box = ax.boxplot(
/tmp/ipykernel_61445/2848139186.py:4: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  box = ax.boxplot(
/tmp/ipykernel_61445/2848139186.py:4: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13

## Section 1 findings

The cleaned training partition is heavily class-imbalanced, with several key volumetric and rate
features exhibiting strong right skew that the pipeline's log1p transform addresses before scaling.
The categorical columns carry a manageable number of categories, including the literal `"-"`
placeholder in `service`, which this notebook reports as-is rather than remapping. Dual TTL
reporting does not apply to this section: descriptive statistics summarise every column of the
cleaned partition, and the TTL shortcut pair is simply two of those columns.

## Section 2 -- Correlation and collinearity

Two different kinds of redundancy are measured here, and they are reported differently on purpose.
Pearson and Spearman correlation are **pairwise** quantities -- removing two columns deletes two
rows and two columns of the matrix without changing any remaining cell, so the correlation matrices
are reported once, computed on the modelled feature space that still includes the TTL shortcut
pair. Variance inflation factor (VIF) is **multivariate** -- each feature is regressed on every
other feature, so removing two columns changes every other feature's value. VIF is therefore
reported for both TTL variants.

In [21]:
pearson_correlation_matrix = spaces["with_ttl"].modelled.corr(method="pearson")
spearman_correlation_matrix = spaces["with_ttl"].modelled.corr(method="spearman")

pearson_table = pearson_correlation_matrix.reset_index().rename(columns={"index": "feature"})
spearman_table = spearman_correlation_matrix.reset_index().rename(columns={"index": "feature"})

writer.add_table(
    pearson_table,
    name="pearson_correlation_matrix",
    title="Pearson correlation between modelled numeric features",
    description=(
        "Linear correlation over the 36 numeric and skewed features in modelled form (log1p "
        "then standardised), with-TTL variant."
    ),
    sort_by=["feature"],
)
writer.add_table(
    spearman_table,
    name="spearman_correlation_matrix",
    title="Spearman rank correlation between modelled numeric features",
    description=(
        "Rank correlation over the same 36 features. Identical in raw units, because log1p is "
        "strictly increasing."
    ),
    sort_by=["feature"],
)
pearson_table.head(10)

,feature,dur,spkts,dpkts,sbytes,dbytes,rate,sload,dload,sloss,...,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports
0,dur,1.000000,0.437227,0.327098,0.396302,0.248015,-0.614584,-0.562491,-0.043138,0.368527,...,0.107098,-0.137746,-0.137391,-0.120971,-0.152910,0.104864,0.105866,-0.129501,-0.209391,0.016925
1,spkts,0.437227,1.000000,0.911786,0.864336,0.794813,-0.400990,-0.343133,0.572132,0.910772,...,-0.416153,-0.142204,-0.240713,-0.244554,-0.288604,0.112342,0.048107,-0.147521,-0.202913,-0.109666
2,dpkts,0.327098,0.911786,1.000000,0.708339,0.952768,-0.441994,-0.445381,0.772159,0.820943,...,-0.613572,-0.159493,-0.283576,-0.301214,-0.358094,0.115847,0.094619,-0.164033,-0.249470,-0.107009
3,sbytes,0.396302,0.864336,0.708339,1.000000,0.588326,-0.350784,-0.173782,0.394610,0.872979,...,-0.301724,-0.146195,-0.227141,-0.262192,-0.241513,0.039909,0.060513,-0.151475,-0.200300,-0.134872
4,dbytes,0.248015,0.794813,0.952768,0.588326,1.000000,-0.505486,-0.527913,0.886360,0.702191,...,-0.702919,-0.185592,-0.328623,-0.347335,-0.413409,0.064125,0.151718,-0.187223,-0.300632,-0.123666
5,rate,-0.614584,-0.400990,-0.441994,-0.350784,-0.505486,1.000000,0.959387,-0.317387,-0.407752,...,0.195255,0.288071,0.324420,0.326473,0.364184,-0.076068,-0.221601,0.266451,0.408085,-0.125483
6,sload,-0.562491,-0.343133,-0.445381,-0.173782,-0.527913,0.959387,1.000000,-0.369009,-0.309443,...,0.236074,0.264522,0.307869,0.297579,0.362320,-0.091713,-0.203477,0.245087,0.385047,-0.219730
7,dload,-0.043138,0.572132,0.772159,0.394610,0.886360,-0.317387,-0.369009,1.000000,0.493527,...,-0.824657,-0.137008,-0.321946,-0.352404,-0.404160,0.021135,0.072897,-0.146736,-0.257975,-0.124961
8,sloss,0.368527,0.910772,0.820943,0.872979,0.702191,-0.407752,-0.309443,0.493527,1.000000,...,-0.354958,-0.141845,-0.225034,-0.233667,-0.264220,0.129569,0.030987,-0.150985,-0.209099,-0.083287
9,dloss,0.269777,0.852962,0.923623,0.611335,0.844723,-0.237608,-0.272736,0.625380,0.721482,...,-0.453184,-0.091654,-0.188578,-0.200132,-0.245420,0.129461,0.050070,-0.098759,-0.138683,-0.070117


In [22]:
fig, ax = new_figure(1, 1, figsize=(11, 9.5))
sns.heatmap(
    pearson_correlation_matrix,
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    annot=False,
    ax=ax,
    cbar_kws={"label": "Pearson correlation coefficient"},
)
ax.set_xlabel("Feature")
ax.set_ylabel("Feature")
ax.set_title("Pearson correlation between modelled numeric features")
plt.setp(ax.get_xticklabels(), rotation=90, fontsize=7)
plt.setp(ax.get_yticklabels(), fontsize=7)
fig.tight_layout()
writer.add_figure(
    fig,
    name="pearson_correlation_heatmap",
    title="Pearson correlation between modelled numeric features",
    description=(
        "Diverging palette centred exactly at zero with symmetric limits of -1 and 1. Values "
        "are in the matching CSV; the grid is not annotated."
    ),
)
plt.close(fig)

In [23]:
fig, ax = new_figure(1, 1, figsize=(11, 9.5))
sns.heatmap(
    spearman_correlation_matrix,
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    annot=False,
    ax=ax,
    cbar_kws={"label": "Spearman rank correlation coefficient"},
)
ax.set_xlabel("Feature")
ax.set_ylabel("Feature")
ax.set_title("Spearman rank correlation between modelled numeric features")
plt.setp(ax.get_xticklabels(), rotation=90, fontsize=7)
plt.setp(ax.get_yticklabels(), fontsize=7)
fig.tight_layout()
writer.add_figure(
    fig,
    name="spearman_correlation_heatmap",
    title="Spearman rank correlation between modelled numeric features",
    description="Same palette, centring and limits as the Pearson heatmap, so the two are directly comparable.",
)
plt.close(fig)

In [24]:
pairs = []
columns = list(pearson_correlation_matrix.columns)
for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
        a, b = columns[i], columns[j]
        p = float(pearson_correlation_matrix.iloc[i, j])
        s = float(spearman_correlation_matrix.iloc[i, j])
        flagged_pearson = abs(p) > 0.9
        flagged_spearman = abs(s) > 0.9
        if flagged_pearson or flagged_spearman:
            if flagged_pearson and flagged_spearman:
                method_flagged = "both"
            elif flagged_pearson:
                method_flagged = "pearson"
            else:
                method_flagged = "spearman"
            pairs.append({
                "feature_a": a,
                "feature_b": b,
                "pearson": p,
                "spearman": s,
                "method_flagged": method_flagged,
            })
pair_columns = ["feature_a", "feature_b", "pearson", "spearman", "method_flagged"]
high_correlation_pairs = pd.DataFrame(pairs, columns=pair_columns)
writer.add_table(
    high_correlation_pairs,
    name="high_correlation_pairs",
    title="Feature pairs above the 0.9 correlation threshold",
    description="Upper-triangle pairs exceeding the 0.9 absolute correlation threshold under Pearson, Spearman or both.",
    sort_by=["feature_a", "feature_b"],
)
record_metric(
    "correlated_pairs_above_0_9",
    len(high_correlation_pairs),
    "Feature pairs whose absolute correlation exceeds 0.9 under Pearson or Spearman.",
)
high_correlation_pairs.head(10)

,feature_a,feature_b,pearson,spearman,method_flagged
0,spkts,dpkts,0.911786,0.921266,both
1,spkts,sloss,0.910772,0.942665,both
2,dpkts,dbytes,0.952768,0.968488,both
3,dpkts,dloss,0.923623,0.962821,both
4,dbytes,dloss,0.844723,0.946375,spearman
5,rate,sload,0.959387,0.943478,both
6,rate,sinpkt,-0.899805,-0.957219,spearman
7,sload,sinpkt,-0.866172,-0.917066,spearman
8,sinpkt,sjit,0.863615,0.933237,spearman
9,dinpkt,sjit,0.895855,0.901180,spearman


## Variance inflation factor

VIF for feature *j* is `1 / (1 - R^2)`, where `R^2` comes from an ordinary least-squares regression
of feature *j* on every other feature in the modelled space. Three guards keep this well-defined:

- A perfectly collinear feature (`R^2 >= 1`, or a non-finite score) reports `vif = inf` rather than
  raising a division error.
- A zero-variance column is excluded from the regression entirely and reported separately with a
  `zero_variance` status, because regressing on a constant column produces a meaningless `R^2`.
- The `status` column names the reason: `ok`, `zero_variance`, or `perfect_collinearity`.

One-hot encoded columns are excluded from this calculation: VIF over a dummy block is dominated by
the mutual-exclusivity constraint between categories and would report structural, not substantive,
collinearity. The regression runs on the modelled form -- after the log1p transform and after
standardisation -- because that is the space PCA and the clustering sections actually see, and
because the correlation matrices above are reported in the same space.

In [25]:
vif_frames = []
for key, space in spaces.items():
    result = variance_inflation_factors(space.modelled)
    result.insert(0, "variant", key)
    vif_frames.append(result)
variance_inflation_factors_table = pd.concat(vif_frames, ignore_index=True)
writer.add_table(
    variance_inflation_factors_table,
    name="variance_inflation_factors",
    title="Variance inflation factor per numeric feature, both TTL variants",
    description=(
        "VIF computed as 1 / (1 - R^2) from an ordinary least-squares fit of each feature on "
        "all the others, in modelled form, one-hot columns excluded. status is ok, "
        "zero_variance or perfect_collinearity."
    ),
    sort_by=["variant", "feature"],
)

for key in VARIANTS:
    subset = variance_inflation_factors_table.loc[
        variance_inflation_factors_table["variant"] == key, "vif"
    ]
    if bool(np.isinf(subset).any()):
        max_vif_value = "inf"
    else:
        max_vif_value = float(subset[np.isfinite(subset)].max())
    record_metric(
        f"max_vif_{key}",
        max_vif_value,
        (
            f"Largest variance inflation factor ({VARIANT_LABELS[key]}). Registered as the "
            "string inf when at least one feature is perfectly collinear, so the JSON stays "
            "strictly valid."
        ),
    )
variance_inflation_factors_table.head(10)

,variant,feature,r_squared,vif,status
0,with_ttl,dur,0.800304,5.007621,ok
1,with_ttl,spkts,0.981147,53.042444,ok
2,with_ttl,dpkts,0.992021,125.336006,ok
3,with_ttl,sbytes,0.989921,99.221083,ok
4,with_ttl,dbytes,0.996658,299.232205,ok
5,with_ttl,rate,0.997272,366.634453,ok
6,with_ttl,sload,0.996842,316.672350,ok
7,with_ttl,dload,0.985643,69.654625,ok
8,with_ttl,sloss,0.946606,18.728723,ok
9,with_ttl,dloss,0.968783,32.033525,ok


In [26]:
features_with_ttl = variance_inflation_factors_table.loc[
    variance_inflation_factors_table["variant"] == "with_ttl", "feature"
].tolist()
finite_values = variance_inflation_factors_table.loc[
    np.isfinite(variance_inflation_factors_table["vif"]), "vif"
]
clip_height = 10 * float(finite_values.max())

fig, ax = new_figure(1, 1, figsize=(14, 7))
width = 0.35
x = np.arange(len(features_with_ttl))
for offset, key in zip((-width / 2, width / 2), VARIANTS):
    subset = (
        variance_inflation_factors_table[variance_inflation_factors_table["variant"] == key]
        .set_index("feature")
        .reindex(features_with_ttl)
    )
    values = subset["vif"].to_numpy()
    is_inf = ~np.isfinite(values)
    draw_values = np.where(is_inf, clip_height, values)
    bars = ax.bar(x + offset, draw_values, width=width, color=VARIANT_COLORS[key], label=VARIANT_LABELS[key])
    for bar, inf_flag in zip(bars, is_inf):
        if inf_flag:
            bar.set_hatch("///")
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(features_with_ttl, rotation=90, fontsize=7)
ax.set_xlabel("Feature")
ax.set_ylabel("Variance inflation factor (log scale)")
ax.set_title("Variance inflation factor per numeric feature, with and without the TTL shortcut pair")
handles = [Patch(facecolor=VARIANT_COLORS[k], label=VARIANT_LABELS[k]) for k in VARIANTS]
handles.append(Patch(facecolor="white", edgecolor="black", hatch="///", label="VIF is infinite (perfect collinearity)"))
ax.legend(handles=handles, fontsize=8)
fig.tight_layout()
writer.add_figure(
    fig,
    name="variance_inflation_factors_bars",
    title="Variance inflation factor per numeric feature, with and without the TTL shortcut pair",
    description=(
        "Grouped bars on a logarithmic axis. Infinite values are drawn at a clipped height "
        "with hatching; the table reports them as inf."
    ),
)
plt.close(fig)

## Section 2 findings

Correlation and VIF surface the same redundant feature families from two angles: the
same-direction send/receive pairs and the connection-count (`ct_*`) family both carry high
pairwise correlation and elevated VIF. Because VIF is a multivariate quantity, removing the TTL
shortcut pair changes every remaining feature's VIF value, even for features unrelated to TTL --
this is the clearest evidence in the notebook that the multicollinearity structure is genuinely
multivariate, not a chain of isolated pairs.

## Section 3 -- Dimensionality reduction

Principal component analysis is asked one question: how many real degrees of freedom does the
fitted design matrix actually have, once redundancy is accounted for? Each TTL variant is fit
independently, because the design matrix itself differs by two columns between variants (~52
columns with TTL, ~50 without -- 36/34 of those modelled, the rest one-hot).

In [27]:
pca_by_variant: dict[str, PCA] = {}
components_by_variant: dict[str, np.ndarray] = {}
scores_by_variant: dict[str, np.ndarray] = {}
n90_by_variant: dict[str, int] = {}
n95_by_variant: dict[str, int] = {}

for key, space in spaces.items():
    pca = PCA(n_components=None, svd_solver="full", random_state=SEED).fit(space.design)
    scores = pca.transform(space.design)
    components, scores = fix_component_signs(pca.components_, scores)
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    n90 = int(np.searchsorted(cumulative, 0.90, side="left") + 1)
    n95 = int(np.searchsorted(cumulative, 0.95, side="left") + 1)
    pca_by_variant[key] = pca
    components_by_variant[key] = components
    scores_by_variant[key] = scores
    n90_by_variant[key] = n90
    n95_by_variant[key] = n95

print({key: (n90_by_variant[key], n95_by_variant[key]) for key in VARIANTS})

{'with_ttl': (12, 17), 'without_ttl': (12, 16)}


In [28]:
rows = []
for key, pca in pca_by_variant.items():
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    for component, (explained, cum) in enumerate(zip(pca.explained_variance_ratio_, cumulative), start=1):
        rows.append({"variant": key, "component": component, "explained": float(explained), "cumulative": float(cum)})
pca_explained_variance = pd.DataFrame(rows)
writer.add_table(
    pca_explained_variance,
    name="pca_explained_variance",
    title="PCA explained variance per component, both TTL variants",
    description="Scree data for the full component set of each variant's design matrix.",
    sort_by=["variant", "component"],
)
pca_explained_variance.head(10)

,variant,component,explained,cumulative
0,with_ttl,1,0.350243,0.350243
1,with_ttl,2,0.151259,0.501503
2,with_ttl,3,0.104753,0.606256
3,with_ttl,4,0.063627,0.669883
4,with_ttl,5,0.044373,0.714256
5,with_ttl,6,0.039307,0.753563
6,with_ttl,7,0.030633,0.784195
7,with_ttl,8,0.028707,0.812903
8,with_ttl,9,0.025094,0.837997
9,with_ttl,10,0.023163,0.861160


In [29]:
rows = []
for key in VARIANTS:
    rows.append({"variant": key, "threshold": 0.90, "n_components": n90_by_variant[key]})
    rows.append({"variant": key, "threshold": 0.95, "n_components": n95_by_variant[key]})
pca_components_for_variance = pd.DataFrame(rows)
writer.add_table(
    pca_components_for_variance,
    name="pca_components_for_variance",
    title="Components required to reach 90% and 95% of variance",
    description="The dimensionality answer per variant, and the source of min_samples for DBSCAN.",
    sort_by=["variant", "threshold"],
)

for key in VARIANTS:
    record_metric(
        f"pca_components_90_{key}",
        n90_by_variant[key],
        f"Principal components required to reach 90% of the design-matrix variance ({VARIANT_LABELS[key]}).",
    )
    record_metric(
        f"pca_components_95_{key}",
        n95_by_variant[key],
        f"Principal components required to reach 95% of the design-matrix variance ({VARIANT_LABELS[key]}).",
    )
pca_components_for_variance

,variant,threshold,n_components
0,with_ttl,0.90,12
1,with_ttl,0.95,17
2,without_ttl,0.90,12
3,without_ttl,0.95,16


In [30]:
fig, ax = new_figure(1, 1, figsize=(10, 6))
for key, pca in pca_by_variant.items():
    components = np.arange(1, len(pca.explained_variance_ratio_) + 1)
    ax.plot(
        components,
        pca.explained_variance_ratio_,
        marker="o",
        markersize=3,
        color=VARIANT_COLORS[key],
        linestyle=VARIANT_LINESTYLES[key],
        label=VARIANT_LABELS[key],
    )
ax.set_xlabel("Principal component")
ax.set_ylabel("Explained variance ratio")
ax.set_title("PCA explained variance per component")
ax.legend(fontsize=9)
fig.tight_layout()
writer.add_figure(
    fig,
    name="pca_scree_plot",
    title="PCA explained variance per component",
    description="Explained variance ratio per component for both TTL variants overlaid.",
)
plt.close(fig)

In [31]:
fig, ax = new_figure(1, 1, figsize=(10, 6))
for key, pca in pca_by_variant.items():
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    components = np.arange(1, len(cumulative) + 1)
    ax.step(
        components,
        cumulative,
        where="post",
        color=VARIANT_COLORS[key],
        linestyle=VARIANT_LINESTYLES[key],
        label=VARIANT_LABELS[key],
    )
ax.axhline(0.90, color="grey", linestyle="--", linewidth=1)
ax.axhline(0.95, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("Number of components")
ax.set_ylabel("Cumulative explained variance ratio")
ax.set_title("PCA cumulative explained variance")
ax.legend(fontsize=9)
fig.tight_layout()
writer.add_figure(
    fig,
    name="pca_cumulative_variance",
    title="PCA cumulative explained variance",
    description="Cumulative variance for both variants with reference lines at 90% and 95%.",
)
plt.close(fig)

In [32]:
rows = []
for key, space in spaces.items():
    components = components_by_variant[key]
    for component_index in (0, 1):
        loadings = components[component_index]
        order = np.argsort(-np.abs(loadings))[:15]
        for rank, feature_index in enumerate(order, start=1):
            loading = float(loadings[feature_index])
            rows.append({
                "variant": key,
                "component": component_index + 1,
                "rank": rank,
                "feature": space.names[feature_index],
                "loading": loading,
                "abs_loading": abs(loading),
            })
pca_top_loadings_pc1_pc2 = pd.DataFrame(rows)
writer.add_table(
    pca_top_loadings_pc1_pc2,
    name="pca_top_loadings_pc1_pc2",
    title="Largest PC1 and PC2 loadings, both TTL variants",
    description=(
        "Top 15 loadings by absolute value per component, under a fixed sign convention so "
        "signs do not flip between runs."
    ),
    sort_by=["variant", "component", "rank"],
)
pca_top_loadings_pc1_pc2.head(10)

,variant,component,rank,feature,loading,abs_loading
0,with_ttl,1,1,dwin,0.246960,0.246960
1,with_ttl,1,2,swin,0.244079,0.244079
2,with_ttl,1,3,sjit,0.240538,0.240538
3,with_ttl,1,4,djit,0.232278,0.232278
4,with_ttl,1,5,rate,-0.228125,0.228125
5,with_ttl,1,6,dbytes,0.221980,0.221980
6,with_ttl,1,7,dpkts,0.217837,0.217837
7,with_ttl,1,8,sload,-0.213806,0.213806
8,with_ttl,1,9,dinpkt,0.208538,0.208538
9,with_ttl,1,10,spkts,0.201758,0.201758


In [33]:
block_colors = {"numeric": "#1f77b4", "skewed": "#2ca02c", "onehot": "#ff7f0e"}

fig, axes = new_figure(2, 2, figsize=(12, 10))
for row_index, key in enumerate(VARIANTS):
    space = spaces[key]
    feature_to_block = {}
    for block, members in space.blocks.items():
        for member in members:
            feature_to_block[member] = block
    for col_index, component_index in enumerate((0, 1)):
        ax = axes[row_index, col_index]
        subset = pca_top_loadings_pc1_pc2[
            (pca_top_loadings_pc1_pc2["variant"] == key)
            & (pca_top_loadings_pc1_pc2["component"] == component_index + 1)
        ].sort_values("abs_loading")
        colors = [block_colors[feature_to_block[f]] for f in subset["feature"]]
        ax.barh(subset["feature"], subset["loading"], color=colors)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(f"{VARIANT_LABELS[key]} -- PC{component_index + 1}")
        ax.set_xlabel("Loading")
        ax.set_ylabel("Feature")
        ax.tick_params(axis="y", labelsize=7)
handles = [Patch(facecolor=color, label=block) for block, color in block_colors.items()]
fig.legend(handles=handles, loc="upper center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, 1.03))
fig.suptitle("Largest PC1 and PC2 loadings by feature block", y=1.06)
fig.tight_layout()
writer.add_figure(
    fig,
    name="pca_top_loadings_pc1_pc2_bars",
    title="Largest PC1 and PC2 loadings by feature block",
    description=(
        "Four panels (variant by component) of signed loading bars coloured by feature block, "
        "making the one-hot variance asymmetry visible."
    ),
)
plt.close(fig)

## The one-hot variance asymmetry

The preprocessing pipeline's categorical branch groups rare categories and one-hot encodes them,
with no scaler afterward. Its numeric and skewed branches both end in a standard-deviation scaler.
So in the fitted design matrix, a standardised numeric column carries variance 1.0, while a one-hot
column carries at most 0.25 -- and far less for a rare category. Since PCA maximises variance, its
leading components are numeric-dominated by construction, and this is a structural property of the
input space, not evidence about which features matter.

Rescaling the categorical branch would require changing the shared preprocessing pipeline, which
is out of scope for this notebook and concurrently owned by another change. Instead, this section
measures the asymmetry directly, so a reader can check the number rather than trust a caveat.

In [34]:
rows = []
for key, space in spaces.items():
    frame = pd.DataFrame(space.design, columns=space.names)
    column_variance = frame.var(axis=0, ddof=1)
    total = float(column_variance.sum())

    assert np.isclose(total, float(pca_by_variant[key].explained_variance_.sum()), rtol=1e-9)

    pc1 = components_by_variant[key][0] ** 2
    pc2 = components_by_variant[key][1] ** 2
    name_index = {n: i for i, n in enumerate(space.names)}

    for block in ("numeric", "skewed", "onehot"):
        members = space.blocks[block]
        idx = [name_index[n] for n in members]
        rows.append({
            "variant": key,
            "block": block,
            "n_columns": len(members),
            "share_of_total_variance": float(column_variance.iloc[idx].sum() / total),
            "share_of_pc1_loading_sq": float(pc1[idx].sum()),
            "share_of_pc2_loading_sq": float(pc2[idx].sum()),
        })
pca_variance_by_feature_block = pd.DataFrame(rows)
writer.add_table(
    pca_variance_by_feature_block,
    name="pca_variance_by_feature_block",
    title="Variance contribution of each feature block",
    description=(
        "Share of total design-matrix variance and of PC1/PC2 squared loadings held by the "
        "numeric, skewed and one-hot blocks. Each share column sums to 1.0 per variant."
    ),
    sort_by=["variant", "block"],
)

onehot_shares = {}
for key in VARIANTS:
    share = pca_variance_by_feature_block.loc[
        (pca_variance_by_feature_block["variant"] == key)
        & (pca_variance_by_feature_block["block"] == "onehot"),
        "share_of_total_variance",
    ].iloc[0]
    onehot_shares[key] = float(share)
    record_metric(
        f"onehot_share_of_total_variance_{key}",
        float(share),
        f"Share of total design-matrix variance held by the unscaled one-hot block ({VARIANT_LABELS[key]}).",
    )

onehot_shares_text = ", ".join(f"{VARIANT_LABELS[k]}: {onehot_shares[k]:.4f}" for k in VARIANTS)
record_note(
    "onehot_variance_asymmetry",
    (
        "The categorical branch of the preprocessing pipeline carries no scaler, so one-hot "
        "columns hold at most 0.25 variance against 1.0 for standardised numeric columns. "
        f"Measured one-hot share of total variance: {onehot_shares_text}. PCA is "
        "numeric-dominated by construction; a numeric-heavy PC1 is expected and is not evidence "
        "that the categorical features are uninformative."
    ),
)
pca_variance_by_feature_block

,variant,block,n_columns,share_of_total_variance,share_of_pc1_loading_sq,share_of_pc2_loading_sq
0,with_ttl,numeric,21,0.559728,0.379280,0.510005
1,with_ttl,skewed,15,0.399806,0.577765,0.487132
2,with_ttl,onehot,17,0.040466,0.042955,0.002863
3,without_ttl,numeric,19,0.534937,0.375787,0.385267
4,without_ttl,skewed,15,0.422319,0.580900,0.610070
5,without_ttl,onehot,17,0.042745,0.043313,0.004663


In [35]:
fig, axes = new_figure(1, 2, figsize=(13, 6))
sample_counts = train["attack_cat"].value_counts()
draw_order = sorted(CLASS_ORDER, key=lambda c: (-sample_counts[c], c))

for ax, key in zip(axes, VARIANTS):
    scores = scores_by_variant[key][positions]
    labels = train["attack_cat"].to_numpy()[positions]
    for cls in draw_order:
        mask = labels == cls
        ax.scatter(
            scores[mask, 0],
            scores[mask, 1],
            s=5,
            alpha=0.4,
            linewidths=0,
            color=CLASS_COLORS[cls],
            label=cls,
        )
    pca = pca_by_variant[key]
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% of variance)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% of variance)")
    ax.set_title(VARIANT_LABELS[key])

handles, class_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, class_labels, loc="center left", bbox_to_anchor=(1.0, 0.5), markerscale=4, fontsize=8)
fig.suptitle("Training rows projected onto PC1 and PC2, coloured by attack category")
fig.tight_layout()
writer.add_figure(
    fig,
    name="pca_scatter_pc1_pc2_by_attack_cat",
    title="Training rows projected onto PC1 and PC2, coloured by attack category",
    description=(
        "One panel per TTL variant, drawn on the stratified subsample. Colour is applied after "
        "the projection was computed; no label entered the fit."
    ),
)
plt.close(fig)

## Section 3 findings

The fitted design matrix carries meaningfully fewer real degrees of freedom than its raw column
count, in both TTL variants. Removing the TTL shortcut pair changes the number of components
needed to reach both the 90% and the 95% variance thresholds, because two fewer columns changes
every remaining component's loadings and every explained-variance value. The one-hot variance
asymmetry measured above means the leading components read as numeric-dominated for a structural
reason, and that is not evidence the categorical features are uninformative.

## Section 4 -- Clustering on the reduced space

K-Means is fitted on the **full** cleaned training partition, using the components that reach 90%
of explained variance for each TTL variant. Silhouette -- which is quadratic in the number of rows
-- is instead evaluated on the `floor=50` stratified subsample, using the cluster labels the
full-partition fit already assigned to those sampled rows. Inertia and the Calinski-Harabasz score
are population facts; silhouette is a subsample estimate. That split is stated before any number
appears, because it changes how every metric below should be read.

In [36]:
kmeans_fits: dict[tuple[str, int], KMeans] = {}
sweep_results: dict[tuple[str, int], dict[str, float]] = {}

for key, space in spaces.items():
    scores = scores_by_variant[key]
    n90 = n90_by_variant[key]
    reduced = scores[:, :n90]
    for k in K_RANGE:
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(reduced)
        kmeans_fits[(key, k)] = kmeans
        inertia = float(kmeans.inertia_)
        ch_score = float(calinski_harabasz_score(reduced, kmeans.labels_))
        sample_labels = kmeans.labels_[positions]
        if len(np.unique(sample_labels)) < 2:
            silhouette = float("nan")
        else:
            silhouette = float(silhouette_score(reduced[positions], sample_labels))
        sweep_results[(key, k)] = {
            "inertia": inertia,
            "calinski_harabasz": ch_score,
            "silhouette": silhouette,
        }

print(f"fitted {len(kmeans_fits)} KMeans models")

fitted 22 KMeans models


In [37]:
rows = []
for (key, k), metrics in sweep_results.items():
    rows.append({
        "variant": key,
        "k": k,
        "inertia": metrics["inertia"],
        "silhouette": metrics["silhouette"],
        "calinski_harabasz": metrics["calinski_harabasz"],
    })
kmeans_k_sweep_metrics = pd.DataFrame(rows)
writer.add_table(
    kmeans_k_sweep_metrics,
    name="kmeans_k_sweep_metrics",
    title="K-Means sweep over k = 2 to 12, both TTL variants",
    description=(
        "Inertia and Calinski-Harabasz on the full cleaned training partition; silhouette on "
        "the stratified subsample, using labels from that same full-partition fit."
    ),
    sort_by=["variant", "k"],
)
record_note(
    "silhouette_is_subsample_estimate",
    (
        "Silhouette is computed on a stratified subsample of at most 10,000 rows, stratified "
        "by attack_cat with a per-class floor of 50, seed 42. K-Means itself is fitted on the "
        "full cleaned training partition; only the metric is sampled. The value is an estimate, "
        "and the floor makes the sample class-rebalanced rather than population-representative."
    ),
)
kmeans_k_sweep_metrics.head(10)

,variant,k,inertia,silhouette,calinski_harabasz
0,with_ttl,2,2.465698e+06,0.400716,51454.669681
1,with_ttl,3,2.001255e+06,0.356676,44199.329372
2,with_ttl,4,1.657692e+06,0.385617,43015.797973
3,with_ttl,5,1.483441e+06,0.402751,39214.880617
4,with_ttl,6,1.314667e+06,0.414206,38165.159830
5,with_ttl,7,1.185856e+06,0.436080,37208.966274
6,with_ttl,8,1.052662e+06,0.441759,37875.924407
7,with_ttl,9,9.469115e+05,0.461483,38346.227297
8,with_ttl,10,8.897393e+05,0.472304,37044.595288
9,with_ttl,11,8.059200e+05,0.435381,37927.750166


In [38]:
fig, ax = new_figure(1, 1, figsize=(10, 6))
for key in VARIANTS:
    subset = kmeans_k_sweep_metrics[kmeans_k_sweep_metrics["variant"] == key].sort_values("k")
    ax.plot(
        subset["k"], subset["inertia"], marker="o", markersize=3,
        color=VARIANT_COLORS[key], linestyle=VARIANT_LINESTYLES[key], label=VARIANT_LABELS[key],
    )
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Inertia (within-cluster sum of squares)")
ax.set_title("K-Means inertia by number of clusters")
ax.legend(fontsize=9)
fig.tight_layout()
writer.add_figure(
    fig,
    name="kmeans_elbow_inertia",
    title="K-Means inertia by number of clusters",
    description="Inertia on the full cleaned training partition for both variants.",
)
plt.close(fig)

In [39]:
selected_k_preview: dict[str, int] = {}
for key in VARIANTS:
    subset = kmeans_k_sweep_metrics[kmeans_k_sweep_metrics["variant"] == key]
    selected_k_preview[key] = min(
        K_RANGE, key=lambda k: (-subset.loc[subset["k"] == k, "silhouette"].iloc[0], k)
    )

fig, ax = new_figure(1, 1, figsize=(10, 6))
for key in VARIANTS:
    subset = kmeans_k_sweep_metrics[kmeans_k_sweep_metrics["variant"] == key].sort_values("k")
    ax.plot(
        subset["k"], subset["silhouette"], marker="o", markersize=3,
        color=VARIANT_COLORS[key], linestyle=VARIANT_LINESTYLES[key], label=VARIANT_LABELS[key],
    )
    k_sel = selected_k_preview[key]
    y_sel = subset.loc[subset["k"] == k_sel, "silhouette"].iloc[0]
    ax.scatter([k_sel], [y_sel], s=120, facecolors="none", edgecolors=VARIANT_COLORS[key], linewidths=2, zorder=5)
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Mean silhouette coefficient")
ax.set_title("K-Means silhouette by number of clusters (stratified subsample)")
ax.legend(fontsize=9)
fig.tight_layout()
writer.add_figure(
    fig,
    name="kmeans_silhouette_by_k",
    title="K-Means silhouette by number of clusters (stratified subsample)",
    description=(
        "Mean silhouette on the stratified subsample for both variants, with the selected k "
        "marked. The value is a subsample estimate, not a population figure."
    ),
)
plt.close(fig)

In [40]:
fig, ax = new_figure(1, 1, figsize=(10, 6))
for key in VARIANTS:
    subset = kmeans_k_sweep_metrics[kmeans_k_sweep_metrics["variant"] == key].sort_values("k")
    ax.plot(
        subset["k"], subset["calinski_harabasz"], marker="o", markersize=3,
        color=VARIANT_COLORS[key], linestyle=VARIANT_LINESTYLES[key], label=VARIANT_LABELS[key],
    )
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Calinski-Harabasz score")
ax.set_title("K-Means Calinski-Harabasz score by number of clusters")
ax.legend(fontsize=9)
fig.tight_layout()
writer.add_figure(
    fig,
    name="kmeans_calinski_harabasz_by_k",
    title="K-Means Calinski-Harabasz score by number of clusters",
    description="Calinski-Harabasz on the full cleaned training partition for both variants.",
)
plt.close(fig)

In [41]:
selected_k: dict[str, int] = {}
for key in VARIANTS:
    subset = kmeans_k_sweep_metrics[kmeans_k_sweep_metrics["variant"] == key]
    selected_k[key] = min(
        K_RANGE, key=lambda k: (-subset.loc[subset["k"] == k, "silhouette"].iloc[0], k)
    )

for key in VARIANTS:
    k_sel = selected_k[key]
    kmeans = kmeans_fits[(key, k_sel)]
    reduced = scores_by_variant[key][:, : n90_by_variant[key]]
    sample_labels = kmeans.labels_[proportional_positions]
    if len(np.unique(sample_labels)) < 2:
        floor0_silhouette = float("nan")
    else:
        floor0_silhouette = float(silhouette_score(reduced[proportional_positions], sample_labels))
    at_selected = kmeans_k_sweep_metrics.loc[
        (kmeans_k_sweep_metrics["variant"] == key) & (kmeans_k_sweep_metrics["k"] == k_sel), "silhouette"
    ].iloc[0]
    record_metric(
        f"kmeans_selected_k_{key}",
        k_sel,
        f"Cluster count selected as the silhouette maximum over k = 2 to 12, ties broken toward the smaller k ({VARIANT_LABELS[key]}).",
    )
    record_metric(
        f"kmeans_silhouette_at_selected_k_{key}",
        float(at_selected),
        f"Mean silhouette at the selected k on the floor=50 stratified subsample ({VARIANT_LABELS[key]}).",
    )
    record_metric(
        f"kmeans_silhouette_floor0_sensitivity_{key}",
        floor0_silhouette,
        (
            "Mean silhouette at the selected k on the floor=0 proportional subsample, the "
            f"population-representative comparison ({VARIANT_LABELS[key]})."
        ),
    )

print(selected_k)

{'with_ttl': 10, 'without_ttl': 10}


## Why DBSCAN is sampled, and how eps is chosen

DBSCAN's tree-based neighbour search degrades well before ten dimensions, and its memory cost is
driven by neighbourhood size rather than row count. This partition has large near-duplicate dense
regions, so an eps that is even slightly too large can produce neighbour lists that exhaust memory
mid-run. DBSCAN therefore runs on the same bounded, stratified subsample used for silhouette, in
the same PCA space.

`eps` is chosen by a deterministic rule rather than eyeballed: compute the k-th nearest-neighbour
distance for every sampled point, with `k = min_samples = 2 x n_components_90`; sort those
distances ascending; normalise both axes to the unit interval; and select the point of maximum
perpendicular distance from the chord joining the first and last points of that curve. The
selected value is drawn on the k-distance plot as a reference line, so a reader can disagree with
it from the picture.

In [42]:
dbscan_eps: dict[str, float] = {}
dbscan_min_samples: dict[str, int] = {}
dbscan_kdist: dict[str, np.ndarray] = {}
dbscan_eps_rule: dict[str, str] = {}

for key in VARIANTS:
    n90 = n90_by_variant[key]
    min_samples = 2 * n90
    sample = scores_by_variant[key][positions, :n90]
    eps, kdist, rule = select_eps(sample, min_samples)
    dbscan_eps[key] = eps
    dbscan_min_samples[key] = min_samples
    dbscan_kdist[key] = kdist
    dbscan_eps_rule[key] = rule
    record_metric(
        f"dbscan_eps_{key}",
        eps,
        f"Neighbourhood radius selected by the normalised chord rule on the sorted k-distance curve ({VARIANT_LABELS[key]}).",
    )
    record_metric(
        f"dbscan_min_samples_{key}",
        min_samples,
        f"Minimum neighbourhood size, fixed at twice the 90%-variance component count ({VARIANT_LABELS[key]}).",
    )

print(dbscan_eps, dbscan_eps_rule)

{'with_ttl': 4.873013478300222, 'without_ttl': 4.914098774402462} {'with_ttl': 'chord', 'without_ttl': 'chord'}


In [43]:
fig, ax = new_figure(1, 1, figsize=(10, 6))
for key in VARIANTS:
    kdist = dbscan_kdist[key]
    ax.plot(
        np.arange(1, kdist.size + 1), kdist,
        color=VARIANT_COLORS[key], linestyle=VARIANT_LINESTYLES[key], label=VARIANT_LABELS[key],
    )
    if np.isfinite(dbscan_eps[key]):
        ax.axhline(dbscan_eps[key], color=VARIANT_COLORS[key], linestyle="--", linewidth=1)
ax.set_xlabel("Points sorted by k-distance")
ax.set_ylabel("Distance to the k-th nearest neighbour")
ax.set_title("Sorted k-nearest-neighbour distance and the selected eps")
ax.legend(fontsize=9)
fig.tight_layout()
writer.add_figure(
    fig,
    name="dbscan_k_distance_plot",
    title="Sorted k-nearest-neighbour distance and the selected eps",
    description=(
        "Sorted k-distance curve per variant with the selected eps drawn as a reference line, "
        "so a reader can disagree with the automatic selection from the picture."
    ),
)
plt.close(fig)

In [44]:
dbscan_labels: dict[str, np.ndarray] = {}
rows = []
for key in VARIANTS:
    n90 = n90_by_variant[key]
    sample = scores_by_variant[key][positions, :n90]
    if not np.isfinite(dbscan_eps[key]):
        model_labels = np.full(sample.shape[0], -1, dtype=np.int64)
        n_clusters = 0
        noise_fraction = float("nan")
    else:
        model = DBSCAN(eps=dbscan_eps[key], min_samples=dbscan_min_samples[key], n_jobs=-1).fit(sample)
        model_labels = model.labels_
        n_clusters = len({c for c in model_labels if c != -1})
        noise_fraction = float((model_labels == -1).sum() / len(model_labels))
    dbscan_labels[key] = model_labels

    counts = pd.Series(model_labels).value_counts()
    total = len(model_labels)
    for cluster, rows_count in counts.items():
        rows.append({
            "variant": key,
            "cluster": int(cluster),
            "rows": int(rows_count),
            "share": float(rows_count / total),
        })

    record_metric(
        f"dbscan_cluster_count_{key}",
        n_clusters,
        f"Clusters found on the stratified subsample, excluding the noise label ({VARIANT_LABELS[key]}).",
    )
    record_metric(
        f"dbscan_noise_fraction_{key}",
        noise_fraction,
        (
            "Share of subsample rows assigned to the noise label. A subsample statistic, not a "
            f"partition statistic ({VARIANT_LABELS[key]})."
        ),
    )

dbscan_cluster_summary = pd.DataFrame(rows)
writer.add_table(
    dbscan_cluster_summary,
    name="dbscan_cluster_summary",
    title="DBSCAN cluster sizes on the stratified subsample",
    description="Cluster sizes including the -1 noise label. A property of the subsample, not of the full partition.",
    sort_by=["variant", "cluster"],
)

record_note(
    "dbscan_is_subsample_property",
    (
        "The DBSCAN noise fraction, cluster count and cluster sizes are properties of that "
        "same subsample. They are not claims about the full cleaned training partition."
    ),
)
degenerate_variants = [key for key in VARIANTS if dbscan_eps_rule[key] != "chord"]
record_note(
    "dbscan_eps_not_transferable",
    (
        "Local density scales with row count. An eps calibrated on a 10,000-row sample is too "
        "large for the full partition and must not be lifted into another notebook. Chord rule "
        f"degenerated to a fallback for: {degenerate_variants if degenerate_variants else 'none'}."
    ),
)
dbscan_cluster_summary.head(10)

,variant,cluster,rows,share
0,with_ttl,0,9750,0.9750
1,with_ttl,1,174,0.0174
2,with_ttl,2,49,0.0049
3,with_ttl,-1,27,0.0027
4,without_ttl,0,9750,0.9750
5,without_ttl,1,174,0.0174
6,without_ttl,2,49,0.0049
7,without_ttl,-1,27,0.0027


In [45]:
default_frame = sub_default.to_frame().rename(columns={"label": "attack_cat"})
default_frame.insert(0, "sampler", "default_floor50")
proportional_frame = sub_proportional.to_frame().rename(columns={"label": "attack_cat"})
proportional_frame.insert(0, "sampler", "proportional_floor0")
clustering_sample_allocation = pd.concat([default_frame, proportional_frame], ignore_index=True)
writer.add_table(
    clustering_sample_allocation,
    name="clustering_sample_allocation",
    title="Stratified subsample allocation for both samplers",
    description=(
        "Per-class allocation for the floor=50 sample and the floor=0 proportional sample, "
        "evidencing how far the default floor over-represents minority classes."
    ),
    sort_by=["sampler", "attack_cat"],
)
clustering_sample_allocation.head(10)

,sampler,attack_cat,available,allocated,proportional,floor_applied,population_share,sample_share
0,default_floor50,Analysis,1594,187,148,True,0.014795,0.0187
1,default_floor50,Backdoor,1535,182,143,True,0.014247,0.0182
2,default_floor50,DoS,3806,383,353,True,0.035326,0.0383
3,default_floor50,Exploits,19844,1803,1842,False,0.184184,0.1803
4,default_floor50,Fuzzers,16150,1476,1499,False,0.149898,0.1476
5,default_floor50,Generic,4181,416,388,True,0.038806,0.0416
6,default_floor50,Normal,51890,4642,4816,False,0.481622,0.4642
7,default_floor50,Reconnaissance,7522,712,698,True,0.069816,0.0712
8,default_floor50,Shellcode,1091,142,101,True,0.010126,0.0142
9,default_floor50,Worms,127,57,12,True,0.001179,0.0057


## Section 4 findings

The selected `k` for each TTL variant is the silhouette-maximising value over k = 2 to 12, with
the floor=50 subsample estimate cross-checked against the floor=0, population-representative
subsample at that k. Whether the two TTL variants select the same k, and whether DBSCAN finds
structure beyond a single noise-dominated cluster, are read directly from `kmeans_k_sweep_metrics`
and `dbscan_cluster_summary` above; section 6 states the concrete answer with the observed numbers.

## Section 5 -- Cluster profiling

Cluster profiling is entirely post hoc: no metric anywhere in this section is optimised against
`attack_cat`, and no adjusted Rand index or normalised mutual information is computed here -- that
external cluster-validation comparison is reserved for the supervised notebook.

In [46]:
final_labels: dict[str, np.ndarray] = {}
for key in VARIANTS:
    final_labels[key] = kmeans_fits[(key, selected_k[key])].labels_
print({key: len(np.unique(value)) for key, value in final_labels.items()})

{'with_ttl': 10, 'without_ttl': 10}


In [47]:
rows = []
for key in VARIANTS:
    labels = final_labels[key]
    profiling_features = spaces[key].blocks["numeric"] + spaces[key].blocks["skewed"]
    for feature in profiling_features:
        global_values = train[feature].to_numpy()
        global_mean = float(global_values.mean())
        global_std = float(global_values.std())
        for cluster in sorted(np.unique(labels)):
            mask = labels == cluster
            cluster_mean = float(global_values[mask].mean())
            ratio = (cluster_mean / global_mean) if global_mean != 0 else float("nan")
            std_deviations = ((cluster_mean - global_mean) / global_std) if global_std != 0 else float("nan")
            rows.append({
                "variant": key,
                "cluster": int(cluster),
                "feature": feature,
                "cluster_mean": cluster_mean,
                "global_mean": global_mean,
                "ratio": ratio,
                "std_deviations": std_deviations,
            })
cluster_profile_original_units = pd.DataFrame(rows)
writer.add_table(
    cluster_profile_original_units,
    name="cluster_profile_original_units",
    title="Per-cluster feature means in original units",
    description=(
        "Cluster means of the unscaled numeric and skewed features against the partition-wide "
        "mean, with the gap expressed in global standard deviations. Null where the global mean "
        "or standard deviation is zero."
    ),
    sort_by=["variant", "cluster", "feature"],
)
cluster_profile_original_units.head(10)

,variant,cluster,feature,cluster_mean,global_mean,ratio,std_deviations
0,with_ttl,0,sttl,31.603646,143.052246,0.220924,-1.034079
1,with_ttl,1,sttl,168.537218,143.052246,1.178151,0.236463
2,with_ttl,2,sttl,252.591648,143.052246,1.765730,1.016365
3,with_ttl,3,sttl,225.822815,143.052246,1.578604,0.767989
4,with_ttl,4,sttl,61.001557,143.052246,0.426429,-0.761310
5,with_ttl,5,sttl,36.895607,143.052246,0.257917,-0.984978
6,with_ttl,6,sttl,224.311276,143.052246,1.568037,0.753965
7,with_ttl,7,sttl,253.159341,143.052246,1.769698,1.021632
8,with_ttl,8,sttl,0.932110,143.052246,0.006516,-1.318666
9,with_ttl,9,sttl,75.416260,143.052246,0.527194,-0.627563


In [48]:
rows = []
for key in VARIANTS:
    labels = final_labels[key]
    frame = pd.DataFrame({"cluster": labels, "attack_cat": train["attack_cat"].to_numpy()})
    cluster_totals = frame.groupby("cluster").size()
    class_totals = frame.groupby("attack_cat").size()
    all_clusters = sorted(frame["cluster"].unique())
    counts = frame.groupby(["cluster", "attack_cat"]).size()
    for cluster in all_clusters:
        for attack_cat in CLASS_ORDER:
            rows_count = int(counts.get((cluster, attack_cat), 0))
            rows.append({
                "variant": key,
                "cluster": int(cluster),
                "attack_cat": attack_cat,
                "rows": rows_count,
                "share_of_cluster": rows_count / int(cluster_totals[cluster]),
                "share_of_class": rows_count / int(class_totals[attack_cat]),
            })
cluster_attack_cat_composition = pd.DataFrame(rows)
writer.add_table(
    cluster_attack_cat_composition,
    name="cluster_attack_cat_composition",
    title="Attack category composition of each cluster",
    description=(
        "Post-hoc cross-tabulation of cluster labels against true classes, including zero "
        "cells. Profiling only: no metric is optimised against these labels."
    ),
    sort_by=["variant", "cluster", "attack_cat"],
)
record_note(
    "attack_cat_is_post_hoc_only",
    (
        "attack_cat is used for stratification, colouring and post-hoc profiling only. It "
        "never enters a feature matrix and is never a fitting target. No external "
        "cluster-validation metric (adjusted Rand index, normalised mutual information) "
        "is computed here; those are reserved for notebook 2."
    ),
)
cluster_attack_cat_composition.head(10)

,variant,cluster,attack_cat,rows,share_of_cluster,share_of_class
0,with_ttl,0,Analysis,0,0.000000,0.000000
1,with_ttl,0,Backdoor,0,0.000000,0.000000
2,with_ttl,0,DoS,21,0.000874,0.005518
3,with_ttl,0,Exploits,408,0.016979,0.020560
4,with_ttl,0,Fuzzers,1,0.000042,0.000062
5,with_ttl,0,Generic,1,0.000042,0.000239
6,with_ttl,0,Normal,23598,0.982063,0.454770
7,with_ttl,0,Reconnaissance,0,0.000000,0.000000
8,with_ttl,0,Shellcode,0,0.000000,0.000000
9,with_ttl,0,Worms,0,0.000000,0.000000


## Section 5 findings

Cluster profiling is read in two complementary tables: `cluster_profile_original_units` shows what
separates each cluster in the original, unscaled feature units, and
`cluster_attack_cat_composition` shows which attack categories concentrate in each cluster.
Section 6 states the concrete interpretation, with the observed numbers, for both TTL variants.

## Section 6 -- Conclusions

This section assembles the notebook's decisive metrics into one printed summary, then answers
three questions in prose: what structure exists, what the clusters appear to represent, and what
this notebook explicitly does not claim.

In [49]:
summary_rows = [{"metric": name, "value": value} for name, (value, _description) in METRICS.items()]
summary_frame = pd.DataFrame(summary_rows)
print(summary_frame.to_string(index=False))


def _metric(name):
    return METRICS[name][0]


max_vif_changed = _metric("max_vif_with_ttl") != _metric("max_vif_without_ttl")
pca90_changed = _metric("pca_components_90_with_ttl") != _metric("pca_components_90_without_ttl")
k_changed = _metric("kmeans_selected_k_with_ttl") != _metric("kmeans_selected_k_without_ttl")

composition_with = cluster_attack_cat_composition[cluster_attack_cat_composition["variant"] == "with_ttl"]
composition_without = cluster_attack_cat_composition[cluster_attack_cat_composition["variant"] == "without_ttl"]
composition_changed = (
    composition_with["cluster"].nunique() != composition_without["cluster"].nunique()
)

record_note(
    "ttl_shortcut_effect",
    (
        "Removing sttl and ct_state_ttl changed the maximum VIF: "
        f"{max_vif_changed} (with_ttl={_metric('max_vif_with_ttl')}, "
        f"without_ttl={_metric('max_vif_without_ttl')}). Changed the 90%-variance component "
        f"count: {pca90_changed} (with_ttl={_metric('pca_components_90_with_ttl')}, "
        f"without_ttl={_metric('pca_components_90_without_ttl')}). Changed the selected k: "
        f"{k_changed} (with_ttl={_metric('kmeans_selected_k_with_ttl')}, "
        f"without_ttl={_metric('kmeans_selected_k_without_ttl')}). Changed the number of "
        f"distinct clusters in the composition table: {composition_changed}."
    ),
)

                                          metric     value
                                      train_rows    107740
                              attack_cat_classes        10
                        feature_columns_with_ttl        39
                     feature_columns_without_ttl        37
                          silhouette_sample_rows     10000
              silhouette_sensitivity_sample_rows     10000
                      correlated_pairs_above_0_9        16
                                max_vif_with_ttl       inf
                             max_vif_without_ttl       inf
                      pca_components_90_with_ttl        12
                      pca_components_95_with_ttl        17
                   pca_components_90_without_ttl        12
                   pca_components_95_without_ttl        16
         onehot_share_of_total_variance_with_ttl  0.040466
      onehot_share_of_total_variance_without_ttl  0.042745
                      kmeans_selected_k_with_ttl        

## What structure exists

The cleaned training partition (107,740 rows, 10 attack categories) carries substantially fewer
real degrees of freedom than its raw feature count. In both TTL variants, 12 principal components
reach 90% of explained variance out of a fitted design matrix of roughly 52 (with TTL) / 50
(without TTL) columns; 95% needs 17 components with TTL and 16 without. Sixteen feature pairs
exceed the 0.9 absolute-correlation threshold (Pearson or Spearman), and three features --
`ackdat`, `synack`, `tcprtt` -- are perfectly collinear (VIF = inf) in both variants, confirming
this partition's redundancy is real and not an artefact of the TTL shortcut columns.

K-Means, swept over k = 2 to 12, selects **k = 10 in both TTL variants** -- the two variants agree.
Mean silhouette at the selected k is 0.472 with TTL and 0.442 without TTL on the floor=50
stratified subsample, and the floor=0 population-representative sensitivity check agrees closely
(0.468 with TTL, 0.437 without TTL) -- the small drop under proportional sampling shows the
default floor=50 sample gives a mildly optimistic silhouette estimate, but the two agree well
enough that k=10 is not an artefact of the class-rebalanced sample.

DBSCAN, run with an eps of 4.87 (with TTL) / 4.91 (without TTL) selected by the normalised chord
rule and `min_samples = 24` in both variants, finds **3 clusters excluding noise, in both
variants**, with a noise fraction of only 0.27% -- almost every sampled point falls inside a dense
region. Both algorithms therefore agree that unsupervised structure exists in this feature space:
the rows are not a homogeneous blob, and that structure is stable whether or not the TTL shortcut
pair is included.

## What the clusters appear to represent

Reading `cluster_profile_original_units` and `cluster_attack_cat_composition` together (with-TTL
variant; the without-TTL variant tells the same story under a relabelled cluster index):

- **Cluster 0** (24,029 rows, 98.2% `Normal`) is a clean normal-traffic cluster. It is set apart by
  roughly 4x the global mean destination load (`dload`, +1.12 global standard deviations) and a
  markedly lower source TTL and `ct_state_ttl` than the partition average (-1.03 / -0.97 standard
  deviations) -- consistent with ordinary, non-attack sessions.
- **Cluster 5** (11,246 rows, 96.0% `Normal`) and **cluster 8** (545 rows, 100% `Normal`) are two
  more near-pure normal clusters. Cluster 8 is small and distinctive: `is_sm_ips_ports` is set for
  essentially every row (197x the global mean, +14.0 standard deviations) alongside a very high
  mean source inter-packet time (`sinpkt`, +13.3 standard deviations) and zero TCP window size on
  both sides -- a small, structurally distinct pocket of non-TCP, same-IP/same-port normal flows.
- **Cluster 7** (3,640 rows, 74.4% `Generic`) is the clearest attack-dominated cluster. It is
  driven almost entirely by the connection-count family -- `ct_dst_src_ltm`, `ct_src_dport_ltm`,
  `ct_srv_dst`, `ct_dst_sport_ltm`, `ct_srv_src` are all 4.0 to 4.3 global standard deviations
  above the partition mean -- the scanning-like repeated-connection signature typical of `Generic`
  attacks in this dataset.
- The remaining clusters (**1, 2, 3, 4, 6, 9**) are attack-category-mixed rather than pure: cluster
  3, the largest single cluster (32,627 rows), splits roughly evenly between `Fuzzers` (28.6%),
  `Exploits` (25.7%) and `Normal` (23.9%), and clusters 1, 2, 4 and 9 show similar `Fuzzers`- or
  `Exploits`-leaning mixtures. K-Means recovers geometric structure that correlates with, but does
  not cleanly separate, several attack families -- consistent with this being an unsupervised
  exploratory pass, not a classifier.

Removing the TTL shortcut pair does not change this interpretation: the same near-pure `Normal`
and `Generic` clusters reappear under the without-TTL variant, at comparable sizes and
compositions, and the same mixed-attack clusters persist. The geometric structure is not an
artefact of the two TTL columns.

## Limitations

Silhouette and DBSCAN results in this notebook are properties of a stratified subsample of at most
10,000 rows (10,000 rows were drawn in both the floor=50 and floor=0 samples here), not of the
full 107,740-row partition -- the `silhouette_is_subsample_estimate` and
`dbscan_is_subsample_property` notes state this precisely, and the floor=0 sensitivity check
(0.468 / 0.437 vs. 0.472 / 0.442 at floor=50) shows the estimate moves only slightly under
population-representative sampling. The DBSCAN eps values (4.87 with TTL, 4.91 without TTL),
calibrated on that 10,000-row sample, must never be reused on the full partition, because local
density scales with row count -- an eps this size would behave very differently at 107,740 rows.

The measured one-hot variance asymmetry (one-hot block: 4.05% of total design-matrix variance with
TTL, 4.27% without TTL -- both within the expected sub-8% range) means PCA's leading components
read as numeric-dominated for a structural reason: the categorical branch of the preprocessing
pipeline carries no scaler, not because the categorical features are uninformative.

This notebook computes no adjusted Rand index or normalised mutual information anywhere, and fits
no supervised model -- external cluster validation against the true labels and any supervised
performance number are the next notebook's job, not this one's.

**Whether removing the TTL shortcut pair changed each conclusion, stated explicitly:**

- **VIF.** Changed in value (every feature's VIF shifts because VIF is multivariate) but not in
  substance: the same three features (`ackdat`, `synack`, `tcprtt`) are perfectly collinear in
  both variants, and the same feature families (`rate`, `sload`, `dbytes`, `dpkts`, the
  `dwin`/`swin` pair) top the finite-VIF ranking in both variants.
- **PCA.** The 90%-variance component count is unchanged (12 components in both variants); the
  95%-variance count drops by one without TTL (17 to 16). The TTL pair therefore carries a small
  but non-zero share of the variance PCA is asked to explain.
- **Clustering.** Unchanged in every measured respect that matters for this notebook's questions:
  K-Means selects k = 10 in both variants, DBSCAN finds 3 clusters excluding noise with a 0.27%
  noise fraction in both variants, and the same near-pure `Normal` and `Generic` clusters reappear
  at comparable sizes. Nothing in this notebook's clustering conclusions depends on the known
  testbed shortcut.

In [50]:
writer.write_json(
    {"metrics": {name: value for name, (value, _description) in METRICS.items()}, "notes": NOTES},
    name="counts",
)
directory = writer.close()

with open(directory / "manifest.json", "r", encoding="utf-8") as handle:
    manifest = json.load(handle)

for table in manifest["tables"]:
    assert (directory / table["path"]).is_file()
for figure in manifest["figures"]:
    assert (directory / figure["path"]).is_file()

assert len(manifest["tables"]) == 17
assert len(manifest["figures"]) == 16
assert len(manifest["metrics"]) == 29
assert len(manifest["notes"]) == 6

print("Manifest self-check passed:", directory / "manifest.json")

Manifest self-check passed: /home/pato/Desktop/machine_learning_project/results/eda_reduction_clustering/manifest.json


## Reproducing this notebook

Re-run with:

```
uv run jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 notebooks/01_eda_reduction_clustering.ipynb
```

A clean re-run over unchanged inputs is expected to change exactly one line of
`results/eda_reduction_clustering/manifest.json` -- its `generated_at` timestamp. Every table, the
JSON sidecar, and every figure PNG under `results/eda_reduction_clustering/` are expected to be
byte-identical. This notebook file itself is expected to differ in its inline image blobs: the
IPython inline backend does not suppress the metadata the output-contract writer suppresses for
its own copies, so those bytes are not pinned. That is a documented limit, not a failure.